# The model zoo

> Trees, forests, boosting and nearest neighbours — what each one quietly assumes, and why on table-shaped data they still beat neural networks.

Read this chapter at `/learn/07-the-model-zoo/`. Exported from `src/content/chapters/07-the-model-zoo.mdx` — edit there, not here.


You now have the machinery: a model, a loss, an optimiser, and a validation set
that doesn't lie to you. Today we do the second axis of [the map](/map/) — what
*shape* the learned function is allowed to take.

One thing up front, because it saves a lot of wasted effort:

**On ordinary table-shaped data, a method called gradient boosting usually beats
neural networks.**

Not sometimes — usually. If your data looks like a spreadsheet, that's where to
start. The middle of this chapter is about why, because the reasons are concrete
and knowing them makes choosing much easier.

## Everything shares one interface

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=600, noise=0.28, random_state=0)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(X_tr.shape, y_tr.shape)

`fit` and `predict`. That's the entire scikit-learn
API — and it's why comparing five completely different model families is a
for-loop rather than a project.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

models = {
    "logistic regression": LogisticRegression(),
    "decision tree":       DecisionTreeClassifier(max_depth=5, random_state=0),
    "random forest":       RandomForestClassifier(n_estimators=200, random_state=0),
    "gradient boosting":   HistGradientBoostingClassifier(random_state=0),
    "k-nearest (k=15)":    KNeighborsClassifier(n_neighbors=15),
    "SVM (rbf kernel)":    SVC(),
}
fitted = {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    fitted[name] = m
    print(f"{name:22s} train {m.score(X_tr, y_tr):.3f}   valid {m.score(X_va, y_va):.3f}")

Now — the numbers are the boring part. Here's the interesting part.

In [ ]:
import matplotlib.pyplot as plt

xx, yy = np.meshgrid(np.linspace(-1.8, 2.8, 250), np.linspace(-1.3, 1.8, 250))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(2, 3, figsize=(9.5, 5.2))
for ax, (name, m) in zip(axes.flat, fitted.items()):
    zz = m.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.28, cmap="coolwarm", levels=1)
    ax.scatter(X_va[:, 0], X_va[:, 1], c=y_va, s=7, cmap="coolwarm", edgecolors="none")
    ax.set_title(f"{name}\nvalid {m.score(X_va, y_va):.3f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()

Please stare at that grid for a minute. It's the most informative picture in the
chapter, and possibly in the week.

You're looking at six different *ways of carving up space*, and each one is
telling you something about what that model can and cannot think.

**Logistic regression** can only draw a straight line. So it fails on two
interleaved crescents, and it will keep failing no matter how long you train it
or how much data you give it. It isn't undertrained. It's incapable.

**The decision tree** draws axis-aligned rectangles — which is why its boundary
is a staircase. It can only ever say "is this feature above or below a number?",
so diagonal boundaries have to be approximated in steps.

**The forest** averages many staircases and gets something noticeably smoother.

**The SVM with an RBF kernel** draws a smooth curve, because that's what its
maths produces.

**k-NN** draws a boundary that follows the data itself, wobbles and all — because
it *is* the data.

Each family has an **inductive bias**: the shape of function it prefers before it
has seen a single data point.

So choosing a model family is choosing an assumption about the world. And that
matters enormously, because when the assumption fits you need very little data —
and when it doesn't, no amount of data will rescue you. Logistic regression will
never learn a crescent, not with a billion examples.

## Decision trees

A tree is nested `if` statements, chosen greedily. At each node it asks a single
question: across every feature and every possible threshold, which one split most
reduces the impurity of the two groups it creates?

Then it does that again on each half. That's the whole algorithm.

In [ ]:
from sklearn.tree import export_text
t = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_tr, y_tr)
print(export_text(t, feature_names=["x0", "x1"]))

That's the *entire model*. Printed. Readable. Auditable. Explainable to a
regulator, a colleague, or your future self at 2am.

No other family on this page gives you that for free, and it's worth more than
people give it credit for.

A fitted tree really is:

`enum Node { Leaf(Class), Split { feature: usize, thresh: f64, lt: Box<Node>, ge: Box<Node> } }`

and inference is exactly the recursive match you'd write without thinking.

Training is the interesting half: a greedy search over `(feature, threshold)`
pairs, scoring each by how much it purifies the children.

And note *greedy* — therefore not optimal. A split that looks mediocre right now
might have enabled two brilliant splits below it, and the tree will never find
out, because it already committed. Finding the truly optimal tree is NP-hard,
which is why nobody does it and everybody uses the greedy version quite happily.

Trees have some lovely practical properties. They're scale-invariant — splitting
on `sqm > 80` doesn't care whether you measured in square metres or square feet.
They handle mixed types naturally. They're unbothered by monotonic
transformations, so taking a log of a feature changes nothing.

They are also **catastrophically prone to overfitting**. An unconstrained tree
will happily grow until every leaf contains exactly one training row.

In [ ]:
for d in [1, 2, 3, 5, 10, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)
    print(f"max_depth {str(d):4s}   train {m.score(X_tr, y_tr):.3f}   valid {m.score(X_va, y_va):.3f}"
          f"   leaves {m.get_n_leaves():4d}")

Perfect training accuracy, mediocre validation accuracy, hundreds of leaves.
That's yesterday's [memorisation](/learn/06-generalisation/) in its purest form —
and this time you can count it, one leaf per memorised row.

## Ensembles: the two ways to combine trees

So if one tree overfits, what do you do?

The answer isn't a better tree. It's *many* trees, combined. And there are
exactly two strategies for combining them, which are — rather satisfyingly —
opposites of each other.

**Bagging (random forest): average many independent overfitters.**

Train each tree on a bootstrap sample of the rows, and let each split consider
only a random subset of the features. Every tree overfits, but every tree
overfits *differently*. Averaging cancels the individual mistakes and keeps only
what they agree on.

**Boosting: build a sequence, each one fixing the last one's mistakes.**

Fit a small tree. Look at what it got wrong. Fit the next tree specifically to
*those residuals*. Repeat a few hundred times, adding each new tree's
contribution scaled down by a small learning rate.

In [ ]:
for n in [1, 5, 25, 200]:
    rf = RandomForestClassifier(n_estimators=n, random_state=0).fit(X_tr, y_tr)
    gb = HistGradientBoostingClassifier(max_iter=n, random_state=0).fit(X_tr, y_tr)
    print(f"{n:3d} trees   forest {rf.score(X_va, y_va):.3f}    boosting {gb.score(X_va, y_va):.3f}")

The difference in *character* is the thing to keep, and it has a practical
consequence you'll use constantly:

A forest can't really overfit by adding trees. More trees just means a better
average of the same thing — so `n_estimators` is a compute budget, not a
hyperparameter. Set it as high as you can afford and stop thinking about it.

Boosting **can** overfit by adding trees, because each new tree is deliberately
chasing the remaining error — and some of the remaining error is noise. So
boosting needs early stopping and forests don't.

XGBoost, LightGBM and CatBoost are all gradient boosting with different
engineering underneath: better handling of missing values, native categorical
features, histogram-based splitting for speed.

scikit-learn's `HistGradientBoostingClassifier` is a LightGBM-style
implementation sitting in the standard library, and it's competitive —
which is why this chapter uses it rather than making you install anything.

Between roughly 2015 and 2020, gradient-boosted trees won the overwhelming
majority of Kaggle competitions on tabular data. They still do. There is a whole
generation of very good data scientists whose main tool is a tree.

## Why trees still beat neural networks on tables

This surprises people who've absorbed the "deep learning solved
everything" story. The reasons are concrete, and I think they're worth
understanding rather than memorising.

**Tabular features have no geometry.**

A convolutional network assumes nearby pixels are related — and they are. A
transformer assumes nearby tokens are related — and they are. But in a
spreadsheet, column 3 and column 4 have no relationship whatsoever. The ordering
is an accident of whoever wrote the CSV.

So the neural network has no structural prior to exploit. It starts from further
back and has to learn from scratch what a tree gets for free.

**Trees handle irregularity natively.** Skewed distributions, outliers, missing
values, wildly different scales, features that only matter above a threshold — a
tree just splits and moves on. A neural network needs all of that normalised away
by hand, first, by you.

**The target is often piecewise-constant.** "Approve the loan if income
> X and credit history > Y" is a step function. Trees *are* step functions.
Neural networks have to approximate a step using smooth activations, and they
spend capacity doing it.

**You usually have thousands of rows, not millions.** Deep learning's real
advantage shows up at a scale that most tabular datasets simply never reach.

The practical rule, and it's short:

**If your data is a table, start with gradient boosting. If your data is pixels,
audio, text, or graph structure, start with a neural network.**

That one sentence would meaningfully improve a large number of production
machine learning projects currently underway.

## k-nearest neighbours

Here's a model with no training step at all. It remembers the data. To predict,
it finds the k most similar stored examples and takes a vote.

That's it. There is no fitting. `fit()` is essentially a variable assignment.

In [ ]:
for k in [1, 5, 25, 100]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    print(f"k={k:3d}   train {m.score(X_tr, y_tr):.3f}   valid {m.score(X_va, y_va):.3f}")

`k=1` has perfect training accuracy *by definition* — every point's nearest
neighbour is itself, so it always votes correctly for its own label. That's a
nice little illustration of why training accuracy can be completely
uninformative.

As `k` grows the boundary smooths out, and eventually oversmooths into "always
predict the majority."

Three things make k-NN worth knowing despite its simplicity.

It requires a distance, so features have to be on comparable
scales — otherwise whichever column happens to have the largest numbers
completely dominates the geometry.

It degrades badly in high dimensions, where a strange thing happens: everything
becomes roughly equidistant from everything else, and "nearest" stops meaning
much.

And — this is the reason it earns its place here — it's the conceptual basis of
**vector search**. Every embedding-based retrieval system you've heard of,
including the retrieval half of RAG, is k-NN with a learned distance and a clever
index. You'll build the learned-distance half in chapter 12.

## Support vector machines, briefly

Find the boundary with the widest margin between the classes — and then, the
clever part, compute distances in a high-dimensional space you never actually
construct.

In [ ]:
for kern in ["linear", "poly", "rbf"]:
    m = SVC(kernel=kern).fit(X_tr, y_tr)
    print(f"{kern:7s} kernel   valid {m.score(X_va, y_va):.3f}")

This is one of the beautiful ideas in machine learning, and it takes
one paragraph.

Notice that many algorithms only ever touch the data through
dot products — $x_i \cdot x_j$ and nothing else. They
never need the individual coordinates.

Now suppose you wanted to work in some much richer feature space $\phi(x)$ —
maybe with all the squares and cross-terms, maybe with a thousand dimensions.
You'd need $\phi(x_i) \cdot \phi(x_j)$.

Here's the trick: for well-chosen $\phi$, there's a function $K$ that computes
that dot product **directly from $x_i$ and $x_j$**, without ever building
$\phi(x)$ at all.

And now the part that made me sit up when I first met it. The RBF kernel

$$K(x_i, x_j) = \exp(-\gamma\|x_i - x_j\|^2)$$

corresponds to a $\phi$ with **infinitely many dimensions**.

You get an infinite-dimensional feature space for the price of one exponential.
You never build it, you never store it, you never touch a single one of its
coordinates — and yet you get to work in it.

This was the dominant idea in machine learning from roughly 1995 to 2012, and I
think it deserves a moment of appreciation even though it lost.

Why it lost is instructive, mind you. The kernel matrix is $n \times n$, so a
million examples means a matrix with $10^{12}$ entries — it simply doesn't scale.
And deep learning turned out to *learn* a useful feature map from the data
instead of requiring a human to choose one cleverly in advance.

An infinitely-clever fixed answer lost to a finitely-clever learned one. That
happens a lot in this field, and it's worth remembering when you're tempted to be
clever.

## Which to reach for

<div class="table-scroll">

| Situation | Start with | Why |
|---|---|---|
| Tabular, any size | gradient boosting | Best accuracy per hour of your time, by a distance |
| Tabular, must be explainable | a shallow tree, or logistic regression | You can print the whole model |
| A strong, honest baseline | logistic / linear regression | Two lines, and everything else has to beat it |
| Images, audio, video | pretrained neural network | There's structure in the input to exploit |
| Text | pretrained transformer | Same |
| Very few rows (< 500) | linear model, or k-NN | Anything flexible will just memorise |
| Similarity / retrieval | embeddings + k-NN | This is what a vector database is |

</div>

**"Why does the random forest score differently every time I add trees?"** It
shouldn't, if `random_state` is set — that fixes the bootstrap samples and the
feature subsets. If you're seeing wobble, check you passed it.

**"What's the difference between `n_estimators` and `max_iter`?"** Same idea,
different libraries' names. Forests count trees with `n_estimators`; the boosting
implementation counts boosting rounds with `max_iter`. Both mean "how many trees."

**"`stratify=y` — what does that do?"** It makes the split preserve the class
proportions. Without it, a random split of an imbalanced dataset can hand you a
validation set with almost none of the minority class, and your metric becomes
noise.

**"The SVM took ages on my dataset."** That's the $n \times n$ kernel matrix from
the fold above. SVMs are excellent up to a few tens of thousands of rows and
painful beyond. It's not you.

**"I don't understand what 'impurity' means."** It's a measure of how mixed a
group's labels are. A group that's all one class has zero impurity; a 50/50 group
has maximum impurity. The tree picks the split that most reduces it, which is
just a formal way of saying "the question that best separates the classes."

## Feature importance, and its limits

In [ ]:
from sklearn.datasets import load_breast_cancer
d = load_breast_cancer()
rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(d.data, d.target)
order = np.argsort(rf.feature_importances_)[::-1][:6]
for i in order:
    print(f"{d.feature_names[i]:26s} {rf.feature_importances_[i]:.3f}")

useful. It tells you what to go and collect more of, and what you can
stop collecting entirely.

But treat it with a healthy suspicion, for two reasons.

Built-in importance is **biased toward high-cardinality features**, because a
continuous feature offers far more possible split points than a binary one, and
more chances to look good.

And it **splits credit arbitrarily between correlated features**. If two columns
say almost the same thing, the forest uses each about half the time — so both
look half as important as they are, which can easily read as "neither of these
matters." Drop them both and watch your model collapse.

Feature importance is **not causation**, and it's not far off being a trap.

"Number of support tickets" being your top predictor of churn does not mean
tickets cause churn. It means unhappy people file tickets *and* leave. Removing
the ticket system will not retain a single customer — though it will destroy your
model, and possibly your ability to notice the problem at all.

For something more careful, look at permutation importance (shuffle one column,
see how much the score drops) or SHAP values. Both are considerably more honest.

Both are also still correlational. Causation needs a different toolkit and often
an actual experiment.

In [ ]:
# A small, real, bundled regression dataset — no download required.
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression

d = load_diabetes()
Xd_tr, Xd_va, yd_tr, yd_va = train_test_split(d.data, d.target, test_size=0.3, random_state=0)

# 1. Fit LinearRegression, RandomForestRegressor and HistGradientBoostingRegressor.
#    Report R^2 on validation for each. Does the fanciest model win?
#
# 2. Compute the baseline: R^2 of always predicting yd_tr.mean().
#    (Hint: by definition of R^2, this is 0.0 — verify it, and understand why.)
#
# 3. For the winner, print the top 3 features by importance (or |coef|).
#
# 4. Now add 20 columns of pure noise to Xd. Which model degrades most?

print("replace me")

Question 4 is the real one. Before you run it, predict which model you think will
suffer most — and *why*. Think about what a tree has to do at every single split
when you hand it twenty useless columns.

In [ ]:
rng = np.random.default_rng(0)
models = {
    "linear":   LinearRegression(),
    "forest":   RandomForestRegressor(n_estimators=300, random_state=0),
    "boosting": HistGradientBoostingRegressor(random_state=0),
}
print("clean data")
for name, m in models.items():
    m.fit(Xd_tr, yd_tr)
    print(f"  {name:9s} R^2 {m.score(Xd_va, yd_va):.3f}")

noise_tr = rng.normal(size=(len(Xd_tr), 20))
noise_va = rng.normal(size=(len(Xd_va), 20))
Xn_tr = np.hstack([Xd_tr, noise_tr])
Xn_va = np.hstack([Xd_va, noise_va])

print("\n+ 20 columns of pure noise")
for name, m in models.items():
    m.fit(Xn_tr, yd_tr)
    print(f"  {name:9s} R^2 {m.score(Xn_va, yd_va):.3f}")

Two lessons, and I like both of them.

**The fancy model does not always win.** On 353 training rows with 10 features,
plain linear regression is competitive with — and often beats — both ensembles.

Small data favours strong assumptions. That's the whole story. Reaching
reflexively for boosting is exactly as much a mistake as reaching reflexively for
a neural network; it's just a more fashionable mistake.

**Noise columns hurt, and they hurt unevenly.** Trees have to consider every
column at every split, so useless features actively *dilute the search* — some
splits get spent on noise that happened to look informative in this particular
sample. The tree can't know it was fooled.

Regularised linear models cope better, because the penalty pushes useless
coefficients toward zero and they stop mattering.

Which is why feature selection is still a real activity in 2026, and did not
quietly go away with deep learning. It just moved to a different part of the
stack.

That's the classical toolkit — and it's a good one that will solve a
great many real problems.

Tomorrow we take the other branch: what happens when you stop choosing features
by hand, and let the model learn them for you.